# Fundamentos y simulación básica

Primer recorrido por KineSIM: definición del motor, condiciones experimentales, red cinética, simulación estocástica y análisis de una trayectoria.

> **Nota científica:** KineSIM es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

## Qué aprenderás

Al terminar este laboratorio podrás: (1) configurar el modelo efectivo de kinesina-1, (2) inspeccionar los estados y transiciones de su red cinética, (3) generar una trayectoria individual con el algoritmo estocástico directo de Gillespie y (4) interpretar sus observables.

## Alcance del modelo

KineSIM 0.1.2 representa el motor como una **red cinética reducida**. No calcula la estructura atomística, las posiciones independientes de las dos cabezas ni todos los intermediarios del ciclo real. La posición cambia únicamente cuando se selecciona un evento que tiene un desplazamiento distinto de cero. Los parámetros predeterminados son ilustrativos y no constituyen un ajuste experimental.

En este notebook se usa la concentración de ATP en mol/L, la fuerza en newtons, la temperatura en kelvin, el tiempo en segundos y la posición en nanómetros.

## Instalación desde PyPI

Este notebook utiliza la distribución publicada de **KineSIM**. No modifica `sys.path` ni importa código desde la carpeta `src/`.

- Nombre de distribución: `kinesim` (PyPI no distingue mayúsculas).
- Namespace de importación: `protein_motor_sim` (se conserva por compatibilidad).
- Versión fijada para reproducibilidad: `0.1.2`.

Ejecuta la siguiente celda con acceso a Internet. Si la versión todavía no aparece en PyPI, primero debe publicarse con Twine. Para probar una subida en TestPyPI, consulta las instrucciones de instalación de este README; no mezcles los índices de TestPyPI y PyPI en una misma resolución de dependencias.


In [ ]:
%pip install --quiet "kinesim[notebooks]==0.1.2"


## 1. Objetivo

En este notebook construiremos una simulación individual, inspeccionaremos la red cinética y analizaremos la trayectoria. Las unidades empleadas son: tiempo (s), posición (nm), fuerza (N) y concentración de ATP (mol/L).

## Modelo físico y matemático

El ciclo simplificado se expresa como una red de estados discretos:

$$
\mathrm{bound\_apo}
\xrightarrow{k_{\mathrm{on}}[ATP]}
\mathrm{atp\_bound}
\xrightarrow{k_{\mathrm{hyd}}}
\mathrm{post\_hydrolysis}.
$$

Desde `post_hydrolysis`, el modelo permite un avance, un retroceso o un desprendimiento. El avance y el retroceso regresan a `bound_apo`; el desprendimiento conduce al estado terminal `detached`.

La asociación de ATP se aproxima como una reacción bimolecular bajo concentración constante:

$$
k_{\mathrm{ATP}}=k_{\mathrm{on}}[ATP],
$$

por lo que $k_{\mathrm{on}}$ tiene unidades de M$^{-1}$ s$^{-1}$ y la tasa pseudo-primer orden tiene unidades de s$^{-1}$. Esta ley describe una etapa concreta, no la velocidad global del motor.

In [ ]:
from importlib.metadata import version as distribution_version
print("KineSIM distribution:", distribution_version("kinesim"))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.analysis.observables import analyze_trajectory
from protein_motor_sim.visualization.plots import plot_trajectory
from protein_motor_sim.visualization.molecular_view import plot_kinesin_schematic

## 2. Definir el motor y el entorno

`Kinesin1` proporciona los parámetros y la red de estados. `Environment` define las condiciones experimentales. La fuerza externa positiva se interpreta como una carga que se opone al avance hacia el extremo positivo.

In [ ]:
motor = Kinesin1()
environment = Environment(
    atp_concentration=1.0e-3,  # 1 mM
    external_force_N=0.0,      # N
    temperature_K=310.0,       # K
)
simulator = GillespieSimulator()

print("Motor:", motor.name)
print("Paso nominal (nm):", motor.step_size_nm)
print("Entorno:", environment)
print("Parámetros:", motor.parameters)

### Cómo leer las condiciones experimentales

- `atp_concentration=1e-3` equivale a 1 mmol/L = 1 mM.
- `external_force_N=0.0` indica ausencia de carga externa en este experimento.
- `temperature_K=310.0` fija la temperatura absoluta usada por las leyes de tasa que dependen de $T$.
- `step_size_nm` es el desplazamiento nominal asignado a los eventos de avance y retroceso.

Los valores de `KinesinParameters` son valores de demostración del prototipo. Cambiarlos permite explorar el modelo, pero no equivale a calibrarlo con un experimento.

## 3. Inspeccionar la red de reacciones

La red actual agrupa el mecanismo en tres estados activos (`bound_apo`, `atp_bound`, `post_hydrolysis`) y un estado terminal (`detached`). No representa explícitamente las dos cabezas ni todos los intermediarios estructurales de kinesina.

In [ ]:
network = motor.build_network()
print("Estado inicial:", network.initial_state)
print("\nEstados:")
for name, state in network.states.items():
    print(f"- {name}: {state.description}; terminal={state.is_terminal}")

print("\nTransiciones y tasas disponibles desde cada estado:")
for state_name in network.states:
    print(f"\n{state_name}")
    for transition, rate in network.rates(state_name, environment):
        print(f"  {transition.event_name:20s} -> {transition.target:18s} {rate:10.4f} s^-1")

### Interpretación de la red

La lista de transiciones muestra los eventos que pueden ocurrir desde cada estado y sus tasas actuales, evaluadas para el `Environment` definido arriba. La tasa total de salida de un estado es la suma de las tasas de sus transiciones disponibles:

$$
a_0(S_i)=\sum_{\mu\in\mathrm{salidas}(S_i)}a_\mu.
$$

En `bound_apo` compiten la asociación de ATP y el desprendimiento. En `atp_bound` compiten la hidrólisis y el desprendimiento. En `post_hydrolysis` compiten los pasos hacia adelante, los pasos hacia atrás y el desprendimiento.

**Qué comprobar:** las tasas tienen unidades de s$^{-1}$; la asociación de ATP debe variar con su concentración; el estado `detached` es terminal y no tiene transiciones salientes.

## 4. Ejecutar una trayectoria de Gillespie

La semilla permite reproducir exactamente la misma realización aleatoria. Cambia `t_max` o `seed` y vuelve a ejecutar para explorar otras trayectorias.

In [ ]:
trajectory = simulator.run(
    motor=motor,
    environment=environment,
    t_max=0.5,
    seed=2026,
)

print("Número de registros:", len(trajectory.times_s))
print("Eventos:", trajectory.number_of_events)
print("Estado final:", trajectory.final_state)
print("ATP consumido:", trajectory.atp_consumed_total)
print("Desplazamiento neto (nm):", trajectory.displacement_nm)

## Cómo funciona Gillespie

El algoritmo de Gillespie trata las transiciones como eventos aleatorios. Para las tasas salientes $a_1,\ldots,a_M$ del estado actual, calcula la tasa total:

$$
a_0=\sum_{\mu=1}^{M}a_\mu.
$$

El tiempo de espera hasta el evento siguiente se extrae de una distribución exponencial:

$$
\tau=-\frac{\ln r_1}{a_0},\qquad 0<r_1<1,
$$

y el evento $\mu$ se selecciona con probabilidad:

$$
P(\mu)=\frac{a_\mu}{a_0}.
$$

Después se actualizan el estado, la posición y el contador de ATP, y se recalculan las tasas. La semilla `seed=2026` permite repetir la misma realización aleatoria con la misma versión del código, los mismos parámetros y el mismo entorno de ejecución.

## 5. Analizar y visualizar

La velocidad calculada es la velocidad media neta de la trayectoria registrada. El `run_length_nm` del analizador actual es el valor absoluto del desplazamiento neto; no debe confundirse automáticamente con todas las definiciones experimentales de longitud de recorrido o processividad.

In [ ]:
summary = analyze_trajectory(trajectory)
summary

### Qué significan los observables

| Observable | Definición en KineSIM | Unidad |
|---|---|---|
| Velocidad media neta | $(x_f-x_0)/(t_f-t_0)$ | nm/s |
| Desplazamiento neto | $x_f-x_0$ | nm |
| Longitud de trayectoria | suma de los módulos de todos los incrementos de posición | nm |
| `run_length_nm` | valor absoluto del desplazamiento neto en esta versión | nm |
| ATP consumido | contador incrementado en el evento `atp_hydrolysis` | moléculas |

No confundas la longitud de trayectoria con `run_length_nm`: la primera suma avances y retrocesos; la segunda usa el desplazamiento neto absoluto. Tampoco identifiques automáticamente este observable con todas las definiciones experimentales de processividad.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_trajectory(trajectory, ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
plot_kinesin_schematic(position_nm=trajectory.positions_nm[-1], ax=ax)
plt.show()

In [ ]:
trajectory_table = pd.DataFrame({
    "time_s": trajectory.times_s,
    "position_nm": trajectory.positions_nm,
    "state": trajectory.states,
    "event": trajectory.events,
})
display(trajectory_table.head(12))

## Lectura de resultados y limitaciones

- Los tramos horizontales de la gráfica posición-tiempo corresponden a esperas entre eventos con desplazamiento; no implican que la proteína esté libre de fluctuaciones microscópicas.
- Los saltos de $+8.2$ o $-8.2$ nm corresponden a eventos mecánicos definidos por la red.
- El diagrama molecular es una ilustración; no es una reconstrucción estructural ni una dinámica molecular de dos cabezas.
- El consumo de ATP se registra en la transición `atp_hydrolysis`. El modelo simplificado permite que el motor se desprenda después de hidrolizar ATP y antes de completar un paso; por ello no debe interpretarse como un acoplamiento estequiométrico perfecto en todas las trayectorias.
- La trayectoria es una realización estocástica individual. Para estimar distribuciones y promedios se necesita un conjunto de simulaciones, como el del notebook 03.

## Referencias para profundizar

- Gillespie, D. T. (1977). *Exact stochastic simulation of coupled chemical reactions*. The Journal of Physical Chemistry, 81(25), 2340–2361. [https://doi.org/10.1021/j100540a008](https://doi.org/10.1021/j100540a008)
- Block, S. M. (2007). *Kinesin motor mechanics: Binding, stepping, tracking, gating, and limping*. Biophysical Journal, 92(9), 2986–2995. [https://doi.org/10.1529/biophysj.106.100677](https://doi.org/10.1529/biophysj.106.100677)

## 6. Interpretación

- Gillespie no calcula una trayectoria continua: genera una secuencia de eventos y tiempos aleatorios.
- La posición cambia cuando ocurre un evento con desplazamiento distinto de cero.
- El ATP se contabiliza en el evento `atp_hydrolysis`. En esta red simplificada no existe todavía un acoplamiento termodinámico completo entre cada ATP y cada paso mecánico.
- La figura molecular es esquemática, no una estructura atomística.